# 文本预处理

In [1]:
import collections
import re  # 正则表达式库，用于字符串的模式匹配和分割
from d2l import torch as d2l

In [2]:
d2l.DATA_HUB['time_machine'] = (d2l.DATA_URL + 'timemachine.txt',
                                '090b5e7e70c295757f55df93cb0a180b9691891a')


In [3]:

def read_time_machine():  #@save
    """将时间机器数据集加载到文本行的列表中"""
    with open(d2l.download('time_machine'), 'r') as f:
        lines = f.readlines()
    return [re.sub('[^A-Za-z]+', ' ', line).strip().lower() for line in lines]


In [4]:
lines = read_time_machine()
print(len(lines))
print(lines[0])
print(lines[10])

3221
the time machine by h g wells
twinkled and his usually pale face was flushed and animated the


In [5]:
def tokenize(lines, token='word'):
    if token == 'word':
        # split() 默认以任意空白字符（空格、制表符等）分割
        return [line.split() for line in lines]
    elif token == 'char':
        return [list(line) for line in lines]
    else:
        print('错误：未知词元类型：' + token)

tokens = tokenize(lines)
for i in range(11):
    print(tokens[i])

['the', 'time', 'machine', 'by', 'h', 'g', 'wells']
[]
[]
[]
[]
['i']
[]
[]
['the', 'time', 'traveller', 'for', 'so', 'it', 'will', 'be', 'convenient', 'to', 'speak', 'of', 'him']
['was', 'expounding', 'a', 'recondite', 'matter', 'to', 'us', 'his', 'grey', 'eyes', 'shone', 'and']
['twinkled', 'and', 'his', 'usually', 'pale', 'face', 'was', 'flushed', 'and', 'animated', 'the']


In [10]:
# 下面是对Vocab类和count_corpus函数的解释性重写，增加中文注释，便于理解其工作原理。

class Vocab:  #@save
    """
    文本词表类，用于将字符串词元和索引进行双向映射，同时计算和保存词频信息。

    参数:
        tokens: 用于构建词表的词元列表（可以是1D或2D）。
        min_freq: 包含进入词表的最低词频。
        reserved_tokens: 预留的特殊词元，如['<pad>']等。

    成员:
        idx_to_token: 存储每个索引对应的词元（列表）。
        token_to_idx: 存储每个词元对应的索引（字典）。
        _token_freqs: 存储每个词元及其对应词频（降序）。
    """
    def __init__(self, tokens=None, min_freq=0, reserved_tokens=None):
        if tokens is None:         # 如果没有提供tokens，默认为空列表
            tokens = []
        if reserved_tokens is None: # 如果没有提供预留词元，默认为空列表
            reserved_tokens = []
        # 1. 统计词频，结果是（词元, 词频）元组
        counter = count_corpus(tokens)
        print(counter)
        # 2. 将词频结果按频率降序排序
        self._token_freqs = sorted(counter.items(), key=lambda x: x[1], reverse=True)

        # 3. 初始化词元到索引和索引到词元的映射。未知词元'<unk>'的索引为0
        self.idx_to_token = ['<unk>'] + reserved_tokens
        self.token_to_idx = {token: idx for idx, token in enumerate(self.idx_to_token)}

        # 4. 添加满足最低词频的高频词元到词表映射中
        for token, freq in self._token_freqs:
            if freq < min_freq:
                break      # 低于最低词频时停止添加
            if token not in self.token_to_idx:
                self.idx_to_token.append(token)
                self.token_to_idx[token] = len(self.idx_to_token) - 1

    def __len__(self):
        """返回词表中词元的数量"""
        return len(self.idx_to_token)

    def __getitem__(self, tokens):
        """
        支持用token或token的列表取索引（若为单token返回索引，列表则递归处理）
        未知词元将返回unk索引(通常为0)
        """
        if not isinstance(tokens, (list, tuple)):
            # 返回单个token的索引，若不存在则返回unk
            return self.token_to_idx.get(tokens, self.unk)
        # 对每个token递归取索引
        return [self.__getitem__(token) for token in tokens]

    def to_tokens(self, indices):
        """
        支持用索引或索引列表取词元
        """
        if not isinstance(indices, (list, tuple)):
            return self.idx_to_token[indices]
        return [self.idx_to_token[index] for index in indices]

    @property
    def unk(self):
        """返回未知词元的索引，通常为0"""
        return 0

    @property
    def token_freqs(self):
        """返回(词元, 词频)的有序列表"""
        return self._token_freqs

def count_corpus(tokens):  #@save
    """
    统计语料中各词元的出现频率
    参数:
        tokens: 1D(词元列表)或2D(每行为词元列表的列表)
    返回:
        collections.Counter对象（词元到计数的映射）
    """
    # 若为2D列表则展开为1D列表
    if len(tokens) == 0 or isinstance(tokens[0], list):
        # 将每一行的词元合并到一个大列表中
        tokens = [token for line in tokens for token in line]
    return collections.Counter(tokens)  # 统计每个词元的出现次数

In [11]:
vocab = Vocab(tokens)
print(list(vocab.token_to_idx.items())[:10])

Counter({'the': 2261, 'i': 1267, 'and': 1245, 'of': 1155, 'a': 816, 'to': 695, 'was': 552, 'in': 541, 'that': 443, 'my': 440, 'it': 437, 'had': 354, 'me': 281, 'as': 270, 'at': 243, 'for': 221, 'with': 216, 'but': 204, 'time': 200, 'were': 158, 'this': 152, 'you': 137, 'on': 137, 'then': 134, 'his': 129, 'there': 127, 'he': 123, 'have': 122, 'they': 122, 'from': 122, 'one': 120, 'all': 118, 'not': 114, 'into': 114, 'upon': 113, 'little': 113, 'so': 112, 'is': 106, 'came': 105, 'by': 103, 'some': 94, 'be': 93, 'no': 92, 'could': 92, 'their': 91, 'said': 89, 'saw': 88, 'down': 87, 'them': 86, 'm
……（冗长输出已省略）


In [14]:
# 这段代码的含义是：对于列表 [0, 10] 中的每一个元素 i，依次执行循环体。
# 也就是会分别处理 tokens[0] 和 tokens[10] 两个元素，展示它们的内容及其在词表中的索引
for i in range(10):
    print(tokens[i])            # 打印第i个token
    print(vocab[tokens[i]], '---')     # 打印该token在vocab中的索引

['the', 'time', 'machine', 'by', 'h', 'g', 'wells']
[1, 19, 50, 40, 2183, 2184, 400] ---
[]
[] ---
[]
[] ---
[]
[] ---
[]
[] ---
['i']
[2] ---
[]
[] ---
[]
[] ---
['the', 'time', 'traveller', 'for', 'so', 'it', 'will', 'be', 'convenient', 'to', 'speak', 'of', 'him']
[1, 19, 71, 16, 37, 11, 115, 42, 680, 6, 586, 4, 108] ---
['was', 'expounding', 'a', 'recondite', 'matter', 'to', 'us', 'his', 'grey', 'eyes', 'shone', 'and']
[7, 1420, 5, 2185, 587, 6, 126, 25, 330, 127, 439, 3] ---


In [15]:
def load_corpus_time_machine(max_tokens=-1):
    lines = read_time_machine()
    tokens = tokenize(lines, 'char')
    vocab = Vocab(tokens)
    corpus = [vocab[token] for line in tokens for token in line]
    print(corpus)
    if max_tokens > 0:
        corpus = corpus[:max_tokens]
    return corpus, vocab

corpus, vocab = load_corpus_time_machine()
len(corpus), len(vocab)

Counter({' ': 29927, 'e': 17838, 't': 13515, 'a': 11704, 'i': 10138, 'n': 9917, 'o': 9758, 's': 8486, 'h': 8257, 'r': 7674, 'd': 6337, 'l': 6146, 'm': 4043, 'u': 3805, 'c': 3424, 'f': 3354, 'w': 3225, 'g': 3075, 'y': 2679, 'p': 2427, 'b': 1897, 'v': 1295, 'k': 1087, 'x': 236, 'z': 144, 'j': 97, 'q': 95})
[3, 9, 2, 1, 3, 5, 13, 2, 1, 13, 4, 15, 9, 5, 6, 2, 1, 21, 19, 1, 9, 1, 18, 1, 17, 2, 12, 12, 8, 5, 3, 9, 2, 1, 3, 5, 13, 2, 1, 3, 10, 4, 22, 2, 12, 12, 2, 10, 1, 16, 7, 10, 1, 8, 7, 1, 5, 3, 1, 17, 5, 12, 12, 1, 21, 2, 1, 15, 7, 6, 22, 2, 6, 5, 2, 6, 3, 1, 3, 7, 1, 8, 20, 2, 4, 23, 1, 7, 16, 
……（冗长输出已省略）


(170580, 28)